# Lab 7: tSNE and UMAP

<a target="_blank" href="https://colab.research.google.com/github/drchadvidden/courseMaterials/blob/main/UnsupervisedLearning/Labs/Lab%207/Lab_7.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab Instructions

Run each of the coding cells. For tutorial example cells, understand the commands and check that the outputs make sense. For exercise cells, write your own code where indicated to generate the correct output. Give text explanations where indicated.

### Submission:
Complete the following notebook in order. Once done, save the notebook, print the file as a .pdf, and upload the resulting file to the Canvas course assignment.

### Rubric:
15 total points, 5 points to running tutorial example cells and saving outputs, 10 points for completing exercises.

### Deadline:
Tuesday at midnight after the lab is assigned.

# Tutorial: t-SNE Explained

## Technical Overview of t-SNE

### What Problem Does t-SNE Solve?

In high-dimensional data, distances become difficult to interpret, and meaningful structure is often hidden. The goal of t-SNE (t-distributed Stochastic Neighbor Embedding) is to create a low-dimensional representation (typically 2D) that preserves the local structure of the data.

---

### Step 1: Convert Distances to Probabilities (High-Dimensional Space)

Instead of working directly with distances, t-SNE converts distances into similarities.

For each pair of points $x_i$ and $x_j$, define:

$$
p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / (2 \sigma_i^2)\right)}
{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / (2 \sigma_i^2)\right)}
$$

- $p_{j|i}$ measures how likely point $x_i$ would pick $x_j$ as a neighbor  
- Each point has its own bandwidth $\sigma_i$, chosen using perplexity  

We then symmetrize:

$$
p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}
$$

---

### Step 2: Define Similarities in Low-Dimensional Space

Let $y_i$ be the low-dimensional representation of $x_i$.

t-SNE defines similarities using a Student $t$-distribution:

$$
q_{ij} = \frac{\left(1 + \|y_i - y_j\|^2\right)^{-1}}
{\sum_{k \neq l} \left(1 + \|y_k - y_l\|^2\right)^{-1}}
$$

Key idea:

- Heavy tails prevent points from crowding together  
- Helps separate clusters in low dimensions  

---

### Step 3: Match the Two Distributions

t-SNE finds the embedding by minimizing the Kullback–Leibler divergence:

$$
\mathrm{KL}(P \,\|\, Q) = \sum_{i \neq j} p_{ij} \log \left(\frac{p_{ij}}{q_{ij}}\right)
$$

- $P$: similarities in high-dimensional space  
- $Q$: similarities in low-dimensional space  

This objective emphasizes preserving nearby points.

---

### Optimization

- Uses gradient descent  
- Non-convex (results may vary across runs)  
- Sensitive to initialization  

---

### Key Parameter: Perplexity

Perplexity controls the effective number of neighbors:

$$
\text{Perplexity}(P_i) = 2^{H(P_i)}
$$

- Small perplexity $\rightarrow$ very local structure  
- Large perplexity $\rightarrow$ more global structure  

Typical values: 5 to 50

---

### Important Properties

t-SNE:

- Preserves local neighborhoods  
- Reveals cluster structure clearly  
- Does **not** preserve global distances  
- Does **not** preserve cluster size or density  
- Axes have no inherent meaning  

---

### Intuition Summary

t-SNE works by:

1. Turning distances into probabilities of being neighbors  
2. Creating a similar probability structure in low dimensions  
3. Moving points to match these two structures  

Interpretation:

> Points that are close are likely similar  

Not:

> Distances and geometry are accurate

## t-SNE for swiss roll dataset

The Swiss roll is a classic example of a nonlinear manifold: the data lives in 3D space, but its intrinsic structure is 2-dimensional (a rolled-up sheet).

Linear methods like PCA fail because they try to preserve global linear structure, but the Swiss roll is curved.

This curve is visualized below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

# Generate data
n_samples = 1500
X, t = make_swiss_roll(n_samples=n_samples, noise=0.05)

# 3D plot
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[:,0], X[:,1], X[:,2], c=t, cmap='viridis', s=10)

ax.set_title("Swiss Roll Dataset (3D)")
plt.show()

## Effect of Perplexity in t-SNE

Perplexity is a key parameter in t-SNE that controls the **effective number of neighbors** used to define similarity.

- Small perplexity (e.g., 5): very local → can fragment structure  
- Medium perplexity (e.g., 30): balanced → often most interpretable  
- Large perplexity (e.g., 50): more global → smoother but less detailed  

We run t-SNE multiple times on the same data, changing only the perplexity.

As you compare the plots, look for:

- Do clusters split apart or stay together?  
- Does the embedding look noisy or smooth?  
- How sensitive is the result to this parameter?  

t-SNE preserves **local neighborhoods**, not global geometry, and results may vary slightly across runs.

> Close points are meaningful; large-scale structure is not

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 3, figsize=(18,5))

perplexities = [5, 30, 50]

for ax, p in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=p, random_state=42, max_iter=250)
    X_tsne = tsne.fit_transform(X)

    ax.scatter(X_tsne[:,0], X_tsne[:,1], c=t, cmap='viridis', s=10)
    ax.set_title(f"Perplexity = {p}")

plt.tight_layout()
plt.show()

## UMAP on the Swiss Roll Dataset

UMAP (Uniform Manifold Approximation and Projection) is a nonlinear dimensionality reduction method similar in goal to t-SNE:

- Creates a low-dimensional (usually 2D) embedding  
- Preserves local neighborhood structure  
- Reveals cluster structure in data  

UMAP works in three main steps:

1. **Construct a high-dimensional graph**:  
   Each point $x_i$ is connected to its $k$-nearest neighbors. The edges are weighted using a smooth exponential kernel:

   $$
   w_{ij} = \exp\left(-\frac{\|x_i - x_j\| - \rho_i}{\sigma_i}\right)
   $$

   - $\rho_i$ is the distance to the nearest neighbor (local connectivity adjustment)  
   - $\sigma_i$ is chosen so that the **effective number of neighbors** matches `n_neighbors`  

2. **Compute a fuzzy simplicial set**:  
   The graph is turned into a **probabilistic representation** of the manifold, where edge weights represent the likelihood that points are connected.  

3. **Optimize a low-dimensional embedding**:  
   Find points $y_i$ in 2D that minimize the cross-entropy between high-dimensional and low-dimensional graphs:

   $$
   C = \sum_{i \neq j} w_{ij} \log\frac{w_{ij}}{v_{ij}} + (1 - w_{ij}) \log\frac{1 - w_{ij}}{1 - v_{ij}}
   $$

   - $v_{ij}$ is the low-dimensional similarity (based on a smooth curve like a Student t-like kernel)  
   - Optimization preserves **local neighborhoods** while capturing global structure  

Key parameters:

- **n_neighbors**: Number of neighbors to define local structure  
  - Small → very local, can fragment clusters  
  - Large → more global, smoother embedding  

- **min_dist**: Controls how tightly points are packed in low dimensions  
  - Small → tighter clusters  
  - Large → more spread-out  

UMAP often preserves **more global structure** than t-SNE and is usually faster. Like t-SNE, axes have **no inherent meaning**, and close points are meaningful, but distances between far-apart points are not.  

We will explore how changing `n_neighbors` affects the embedding of the Swiss roll dataset. Look for:

- Preservation of local clusters  
- Smoothness of the overall manifold  
- How global shape emerges as `n_neighbors` increases

In [ ]:
import umap

neighbors_list = [5, 15, 50]  # parameter to vary

fig, axes = plt.subplots(1, 3, figsize=(18,5))

for ax, n in zip(axes, neighbors_list):
    reducer = umap.UMAP(
        n_neighbors=n,
        min_dist=0.1,
        n_components=2,
        n_epochs=100,   # fewer epochs for speed
        random_state=None,
        n_jobs=-1 # parallel computing
    )

    X_umap = reducer.fit_transform(X)

    ax.scatter(X_umap[:,0], X_umap[:,1], c=t, cmap='viridis', s=10)
    ax.set_title(f"n_neighbors = {n}")

plt.tight_layout()
plt.show()

# Exercise(s): tSNE and UMAP





## Exercise 1: PCA on the Swiss Roll

### Tasks:
1. Apply PCA to the Swiss roll and reduce it to 2D.  
2. Plot the 2D embedding, coloring points by `t`.  
3. Compare the PCA embedding to the original 3D roll. What structure is lost?  
4. Write a short note on why PCA fails for this nonlinear manifold.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 2: Reading and Reproducing Results from “How to Use t‑SNE Effectively”

Read the interactive article **“How to Use t‑SNE Effectively”** on *Distill.pub* (linked below). This piece explores common misunderstandings in interpreting t‑SNE plots and illustrates how hyperparameters and algorithm behavior affect the resulting visualizations:

https://distill.pub/2016/misread-tsne/

### Tasks:
1. Write a **brief summary (3–5 sentences)** of the key points in the article. In your summary, make sure to include:
   - Why t‑SNE plots can sometimes be **misleading**  
   - The role of **hyperparameters** (especially perplexity)  
   - What the article says about interpreting **cluster sizes and distances** in t‑SNE plots

2. Choose **one of the example graphs** from the article (e.g., the effect of different perplexity values, cluster size distortion, or random noise) and **reproduce it** with your own code using a synthetic dataset of your choice. Write **1–2 sentences** explaining how your reproduced visualization demonstrates the point made in the article. You can use sklearn tools to generate data:

https://scikit-learn.org/stable/datasets/sample_generators.html


In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 3: UMAP Tutorial (Digits Data)

### Tasks:
1. Follow the UMAP tutorial on digits data:  
   https://umap-learn.readthedocs.io/en/latest/basic_usage.html#digits-data  

2. Reproduce the main result:
   - Load the digits dataset from `sklearn`  
   - Fit a UMAP model  
   - Plot the 2D embedding colored by digit label  

3. Write 2–3 sentences describing what you observe:
   - Are digit classes separated?  
   - Do any digits overlap or form subclusters?  

4. Apply **PCA** to the same dataset and plot the 2D result.  

5. Apply **t-SNE** to the same dataset and plot the 2D result.  

6. Compare all three methods (UMAP, PCA, t-SNE) in 2–3 sentences:
   - Which best separates the digit classes?  
   - How does global structure differ between methods?

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 4: UMAP for Clustering

### Tasks:
1. Follow and reproduce the main workflow from the UMAP clustering tutorial:  
   https://umap-learn.readthedocs.io/en/latest/clustering.html  

   In particular:
   - Load the MNIST dataset  
   - Visualize the UMAP embedding colored by true labels  
   - Apply K-Means and evaluate the clustering  
   - Apply HDBSCAN (with PCA preprocessing)  
   - Apply UMAP + HDBSCAN and visualize the results  

---

2. Compare the three approaches:

- K-Means (high-dimensional data)  
- PCA + HDBSCAN  
- UMAP + HDBSCAN  

Write 3–4 sentences describing:
- Which method performs best  
- What kinds of clusters each method finds  

---

3. Quantitative evaluation:

Use clustering metrics (e.g., Adjusted Rand Index or Adjusted Mutual Information) to compare performance.

- Which method gives the highest score?  
- Does this match what you see visually?  

---

4. Critical thinking:

The tutorial notes that using UMAP for clustering can be **controversial** because it may distort density and create artificial clusters.

Write 2–3 sentences addressing:

- Why might UMAP improve clustering performance?  
- Why might it also be misleading?  

---

5. Exploration (interesting part):

Modify UMAP parameters:

- Increase `n_neighbors` (e.g., 50 or 100)  
- Set `min_dist = 0`  

Re-run clustering and describe:

- Do clusters become more separated or more merged?  
- Does clustering performance improve or degrade?  

---

### Key Idea

UMAP is not a clustering algorithm itself. It is a **preprocessing step** that can make clusters more visible by reducing dimensionality, but it may also introduce artifacts.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 5: t-SNE from Scratch

### Tasks:
1. Read Chapter 7 (t-SNE) from the Ebrahimia notes (see Canvas HW page).  

2. Reproduce the **t-SNE implementation from scratch** as outlined in the chapter. Your implementation should include:
   - Computing pairwise distances  
   - Converting distances to probabilities $P_{ij}$  
   - Defining low-dimensional similarities $Q_{ij}$  
   - Minimizing the KL divergence using gradient descent  

3. Apply your implementation to a simple dataset (e.g., Swiss roll, blobs, or digits) and produce a 2D embedding plot.  

4. Compare your result to `sklearn.manifold.TSNE` on the same dataset.  

5. Write 2–3 sentences addressing:
   - How similar are the embeddings?  
   - What aspects of the algorithm were hardest to implement?  

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




# HTML Export Code

In [ ]:
# code to export notebook as .html for Canvas upload

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

notebook_name = "Lab_7"
!cp "/content/drive/MyDrive/Colab Notebooks/DSC 430/{notebook_name}.ipynb" /content/
!jupyter nbconvert --to html "/content/{notebook_name}.ipynb"
files.download(f"/content/{notebook_name}.html")

